In [2]:
# Checking device
import os
import torch

t = torch.tensor([5, 5, 5], dtype=torch.int64, device='mps')
t_f32 = torch.tensor([5, 5, 5], dtype=torch.float32, device='mps')
print('Testing PyTorch ROCM support...')

print(t, t_f32)

Testing PyTorch ROCM support...
tensor([5, 5, 5], device='mps:0') tensor([5., 5., 5.], device='mps:0')


## GPT-oss:20b

In [9]:
import requests
import time


class OllamaClient:
    def __init__(
        self,
        model: str = "gpt-oss:20b",
        base_url: str = "http://localhost:11434",
        timeout: int = 300,
    ):
        self.model = model
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
    
    def generate(self, prompt: str) -> dict:
        url = f"{self.base_url}/api/generate"
        payload = {"model": self.model, "prompt": prompt, "stream": False}
    
        start_time = time.perf_counter()
        response = requests.post(url, json=payload)
        response.raise_for_status()
        total_wall_time = time.perf_counter() - start_time
        
        data = response.json()
    
        # Metrics from Ollama (in nanoseconds)
        prompt_tokens = data.get("prompt_eval_count", 0)
        output_tokens = data.get("eval_count", 0)
        
        # eval_duration is strictly the generation phase
        eval_duration_ns = data.get("eval_duration", 1) # Avoid div by zero
        
        # Calculate TPS based purely on generation time
        tokens_per_sec = output_tokens / (eval_duration_ns / 1e9)
    
        return {
            "text": data.get("response"),
            "input_tokens": prompt_tokens,
            "output_tokens": output_tokens,
            "total_tokens": prompt_tokens + output_tokens,
            "gen_speed_tps": round(tokens_per_sec, 2),
            "total_latency_sec": round(total_wall_time, 3)
        }


In [10]:
ollama = OllamaClient(model="gpt-oss:20b")


### Reasoning

In [4]:
prompt = """
A bat and a ball cost $1.10 in total. 
The bat costs $1.00 more than the ball. 
How much does the ball cost? 
Think step-by-step and provide the final answer.
"""

result = ollama.generate(prompt=prompt)

print(result["text"])

**Step‑by‑step solution**

1. **Define the variables**  
   Let  
   \[
   \text{ball cost} = x \text{ dollars}
   \]  
   The bat is $1.00 more expensive than the ball, so  
   \[
   \text{bat cost} = x + 1.00 \text{ dollars}
   \]

2. **Set up the total‑cost equation**  
   The problem says the bat and ball together cost $1.10.  
   \[
   x + (x + 1.00) = 1.10
   \]

3. **Simplify the equation**  
   \[
   2x + 1.00 = 1.10
   \]

4. **Solve for \(x\)**  
   Subtract 1.00 from both sides:  
   \[
   2x = 0.10
   \]  
   Divide by 2:  
   \[
   x = 0.05
   \]

5. **Interpret the result**  
   \(x = 0.05\) dollars, which is **5 cents**.

---

**Answer:** The ball costs **$0.05** (five cents).


In [5]:
prompt = """
I put a diamond inside a glass cup. 
I place the glass cup inside a microwave. 
I turn the microwave on for 30 seconds. 
I take the glass cup out and put it in the freezer. 
Where is the diamond now and what is its state?
"""

result = ollama.generate(prompt=prompt)

print(result["text"])

**Location:**  
The diamond is still in the glass cup – now sitting in the freezer.

**State:**  
The diamond remains a solid, crystalline diamond.  
It has not melted, sublimed, or changed phase; its structure is unchanged.  

---

### Why this is the case

| Step | What happens (in short) | Effect on the diamond |
|------|-------------------------|-----------------------|
| 1.  Diamond in a glass cup | The diamond is physically inside the cup, protected from the microwave environment. | No direct interaction. |
| 2.  Cup placed in microwave (30 s) | Microwaves heat polar molecules; glass (non‑polar but can absorb some microwave energy) may warm slightly, but the diamond is an excellent electrical insulator and doesn’t absorb microwave energy. | The diamond may receive a little heat by conduction from the cup, but it stays solid. |
| 3.  Cup taken out & put in freezer | The cup and any absorbed heat cool. | The diamond cools back to ambient temperature. |

**Key points**

- **Microwav

### Logic

In [11]:
prompt = """
Evaluate the logical validity of this argument. Do not use real-world facts, only the provided statements.

Statement 1: All flurgs are green.
Statement 2: Some green things are electric.
Conclusion: Therefore, some flurgs are electric.

Is this conclusion logically valid? Explain why or why not.
"""

result = ollama.generate(prompt=prompt)

print(result["text"])

**No, the conclusion is not logically valid.**

---

### 1.  Translate the argument into predicate logic

| English | Predicate logic |
|---------|-----------------|
| All flurgs are green. | ∀x (Flurg(x) → Green(x)) |
| Some green things are electric. | ∃x (Green(x) ∧ Electric(x)) |
| Therefore, some flurgs are electric. | ∃x (Flurg(x) ∧ Electric(x)) |

The question is whether the premises entail the conclusion.

---

### 2.  Check the entailment

To be valid, **every model** (interpretation) that makes the premises true must also make the conclusion true.  
We can find a model in which the premises are true but the conclusion is false—this shows the argument is **invalid**.

---

#### Counter‑model

| Object | Flurg? | Green? | Electric? |
|--------|--------|--------|-----------|
| a      | No     | Yes    | Yes       |
| b      | No     | Yes    | No        |
| c      | No     | No     | No        |

*Premise 1*  
∃x (Flurg(x) → Green(x)) is true for all x because there are no flurg

In [12]:
prompt = """
You are on an island where there are two types of people: Knights, who always tell the truth, and Knaves, who always lie.

You meet two people, A and B.
A says: "At least one of us is a Knave."

What are A and B? Walk through the logic to find the solution.
"""

result = ollama.generate(prompt=prompt)

print(result["text"])

Let’s translate the situation into a short logical form.

---

### 1.  Translate the sentence

> “At least one of us is a Knave.”

Let  
- \(A\) = “A is a Knave”  
- \(B\) = “B is a Knave”

The sentence is simply  

\[
S \;=\; (A \text{ or } B)
\]

So \(S\) is true exactly when **at least one** of the two people is a Knave.

---

### 2.  Apply the Knights‑and‑Knaves rule

- If a person is a **Knight**, the statement he says must be **true**.  
- If a person is a **Knave**, the statement he says must be **false**.

We only have one statement: A says \(S\).  
Therefore:

| A’s type | What \(S\) must be | What this implies for A and B |
|----------|-------------------|--------------------------------|
| Knight   | \(S\) **true**    | “At least one of us is a Knave” is true. A cannot be a Knave, so B must be a Knave. |
| Knave    | \(S\) **false**   | “At least one of us is a Knave” is false → **neither** is a Knave. That means both A and B are Knights, contradicting that A is a Knave. |



### Code

In [13]:
prompt = """
Write a Python function to solve the "Container With Most Water" problem. 
Given an array of integers representing heights of vertical lines, find two lines that together with the x-axis form a container, such that the container contains the most water.

Requirement: The solution must be O(n) time complexity. Explain your logic before writing the code.
"""

result = ollama.generate(prompt=prompt)

print(result["text"])

**Explanation – Why a Two‑Pointer Scan Works**

The goal is to pick two indices `i` and `j` (`i < j`) that maximize the area

```
area(i, j) = min(height[i], height[j]) * (j – i)
```

A brute‑force double loop would take `O(n²)` time.  
We can do better because:

1. **The limiting factor is the shorter line**  
   For any fixed pair `(i, j)` the height of the water that can be held is bounded by the shorter of the two lines.  
   So if we keep the shorter side and try to find a *taller* line further away, we might increase the area.

2. **Moving the longer side never helps**  
   Suppose `height[i] < height[j]`.  
   If we move the right pointer `j` leftwards (towards `i`), the width `(j – i)` decreases, and the height cannot increase beyond `height[i]` (the shorter side).  
   Therefore any new pair that keeps the left pointer `i` fixed will have an area no larger than the current one.  
   Consequently we **must move the pointer that is on the shorter line**.

By maintaining two indi

In [14]:
prompt = """
Design and implement a Least Recently Used (LRU) Cache in Python. 
It should support two operations: get(key) and put(key, value).
Both operations must happen in O(1) time complexity.

Please provide the complete class implementation and explain which data structures you chose and why.
"""

result = ollama.generate(prompt=prompt)

print(result["text"])

**Solution Overview**

An LRU (Least‑Recently‑Used) cache keeps the most recently accessed items in memory while evicting the oldest items once the cache reaches its capacity.  
The two required operations are

| Operation | Complexity target | What it must do |
|-----------|-------------------|-----------------|
| `get(key)` | O(1) | Return the value for *key* (or `-1`/`None` if missing) **and** mark the key as most‑recently used |
| `put(key, value)` | O(1) | Insert or update the value for *key* and mark it as most‑recently used. If the cache is full, evict the least‑recently used item |

To achieve **O(1)** for both operations we need

1. **Constant‑time lookup of a key → value**  
   → a hash table (`dict` in Python).

2. **Constant‑time update of “recency” order**  
   → a data structure that lets us remove a node from the middle of a list and move it to the front in O(1).  
   → a *doubly‑linked list* is perfect for this because each node knows its previous and next node, so we c

### Latency

In [16]:

print("Tokens:", result["total_tokens"])
print("Latency (s):", result["total_latency_sec"])
print("Tokens/sec:", result["gen_speed_tps"])

Tokens: 2078
Latency (s): 42.633
Tokens/sec: 46.65


## Qwen3:30b

In [17]:
import requests
import time


class OllamaClient:
    def __init__(
        self,
        model: str = "qwen3:30b",
        base_url: str = "http://localhost:11434",
        timeout: int = 300,
    ):
        self.model = model
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
    
    def generate(self, prompt: str) -> dict:
        url = f"{self.base_url}/api/generate"
        payload = {"model": self.model, "prompt": prompt, "stream": False}
    
        start_time = time.perf_counter()
        response = requests.post(url, json=payload)
        response.raise_for_status()
        total_wall_time = time.perf_counter() - start_time
        
        data = response.json()
    
        # Metrics from Ollama (in nanoseconds)
        prompt_tokens = data.get("prompt_eval_count", 0)
        output_tokens = data.get("eval_count", 0)
        
        # eval_duration is strictly the generation phase
        eval_duration_ns = data.get("eval_duration", 1) # Avoid div by zero
        
        # Calculate TPS based purely on generation time
        tokens_per_sec = output_tokens / (eval_duration_ns / 1e9)
    
        return {
            "text": data.get("response"),
            "input_tokens": prompt_tokens,
            "output_tokens": output_tokens,
            "total_tokens": prompt_tokens + output_tokens,
            "gen_speed_tps": round(tokens_per_sec, 2),
            "total_latency_sec": round(total_wall_time, 3)
        }


In [18]:
ollama = OllamaClient(model="qwen3:30b")

### Reasoning

In [20]:
prompt = """
A bat and a ball cost $1.10 in total. 
The bat costs $1.00 more than the ball. 
How much does the ball cost? 
Think step-by-step and provide the final answer.
"""

result = ollama.generate(prompt=prompt)

print(result["text"])

To solve the problem, we begin by defining the cost of the ball as $ x $ dollars. According to the problem:

- The **bat** costs **$1.00 more** than the ball.
- Therefore, the **bat costs** $ x + 1.00 $.
- The **total cost** of both the bat and the ball is **$1.10**.

---

### Step 1: Set up the equation

$$
\text{Cost of ball} + \text{Cost of bat} = 1.10
$$
$$
x + (x + 1.00) = 1.10
$$

---

### Step 2: Simplify the equation

$$
x + x + 1.00 = 1.10
$$
$$
2x + 1.00 = 1.10
$$

---

### Step 3: Solve for $ x $

Subtract 1.00 from both sides:

$$
2x = 0.10
$$

Divide both sides by 2:

$$
x = 0.05
$$

---

### Step 4: Interpret the result

- The **ball costs $0.05** (or 5 cents).
- The **bat costs $0.05 + $1.00 = $1.05**.
- Total: $0.05 + $1.05 = $1.10 ✅

---

### Common Mistake to Avoid

Many people intuitively think the ball costs $0.10, which would make the bat cost $1.10, leading to a total of $1.20 — **not** $1.10. This is why it's important to set up the equation properly instead of r

In [21]:
prompt = """
I put a diamond inside a glass cup. 
I place the glass cup inside a microwave. 
I turn the microwave on for 30 seconds. 
I take the glass cup out and put it in the freezer. 
Where is the diamond now and what is its state?
"""

result = ollama.generate(prompt=prompt)

print(result["text"])

Based on the properties of diamonds, glass, microwaves, and freezing, here's a clear breakdown of what happens:

### 1. **Microwave Step (30 seconds):**
   - **Diamond behavior:** Diamonds are **not affected by microwave radiation**. Microwaves heat materials by exciting *water molecules* or *polar substances*, but diamonds (pure carbon) are non-polar and do not absorb microwave energy. The diamond **remains unchanged**—it does not heat up, melt, or react.
   - **Glass cup behavior:** Standard glass (e.g., borosilicate or soda-lime) is microwave-safe and **does not heat significantly** in a short 30-second cycle. It may feel slightly warm from ambient air, but it won't become hot enough to shatter or alter the diamond.
   - **Result:** The diamond stays intact inside the glass cup. No damage occurs.

### 2. **Freezer Step:**
   - **Diamond behavior:** Diamonds are **extremely stable** across a wide temperature range. Freezing (typically -18°C / 0°F) has **no effect** on a diamond’s str

### Logical

In [22]:
prompt = """
Evaluate the logical validity of this argument. Do not use real-world facts, only the provided statements.

Statement 1: All flurgs are green.
Statement 2: Some green things are electric.
Conclusion: Therefore, some flurgs are electric.

Is this conclusion logically valid? Explain why or why not.
"""

result = ollama.generate(prompt=prompt)

print(result["text"])

The argument is **logically invalid**. Here's why:

1. **Statement 1** ("All flurgs are green") means the set of flurgs is entirely contained within the set of green things.  
2. **Statement 2** ("Some green things are electric") means there exists *at least one* green thing that is electric, but it does **not** specify that this electric green thing is a flurg.  
3. The conclusion ("Some flurgs are electric") requires that **at least one flurg** is electric. However, the premises do not guarantee this. The "some green things" that are electric could be **non-flurgs** (e.g., green electric cars), while all flurgs might still be non-electric.  

**Example to illustrate**:  
- Let "flurgs" = green apples (all green).  
- Let "some green things that are electric" = green electric cars (not apples).  
Here, all flurgs (apples) are green, and some green things (cars) are electric, but **no flurgs are electric**. The premises are true, but the conclusion is false.  

Since the conclusion doe

In [23]:
prompt = """
You are on an island where there are two types of people: Knights, who always tell the truth, and Knaves, who always lie.

You meet two people, A and B.
A says: "At least one of us is a Knave."

What are A and B? Walk through the logic to find the solution.
"""

result = ollama.generate(prompt=prompt)

print(result["text"])

To determine the identities of A and B, we analyze the logical implications of A's statement: **"At least one of us is a Knave."**

---

### Step 1: Understand the Types

- **Knights** always tell the truth.
- **Knaves** always lie.

So, if A says something, it must be **true** if A is a Knight, and **false** if A is a Knave.

---

### Step 2: Analyze A's Statement

A says: **"At least one of us is a Knave."**

This is logically equivalent to saying:  
> **Either A is a Knave, or B is a Knave (or both).**

The **negation** of this statement (i.e., what would be true if A were lying) is:  
> **Neither A nor B is a Knave**, i.e., **both are Knights**.

---

### Step 3: Consider All Possibilities

Let’s test all four possible combinations for A and B:

| A      | B      | Is A's statement true? | A's identity consistent? |
|--------|--------|------------------------|---------------------------|
| Knight | Knight | No (both are Knights)  | ❌ A would be lying → contradiction |
| Knight | Kn

### Code

In [24]:
prompt = """
Write a Python function to solve the "Container With Most Water" problem. 
Given an array of integers representing heights of vertical lines, find two lines that together with the x-axis form a container, such that the container contains the most water.

Requirement: The solution must be O(n) time complexity. Explain your logic before writing the code.
"""

result = ollama.generate(prompt=prompt)

print(result["text"])

To solve the "Container With Most Water" problem efficiently with O(n) time complexity, we use a two-pointer technique that systematically explores potential containers while maximizing the area.

### Approach
1. **Initialization**: Start with two pointers, `left` at the beginning (index 0) and `right` at the end (index `len(heights) - 1`) of the array. This represents the widest possible container.
2. **Area Calculation**: For each pair of lines pointed to by `left` and `right`, calculate the area as the product of the distance between them (`right - left`) and the height of the shorter line (`min(heights[left], heights[right])`).
3. **Update Maximum Area**: Track the maximum area encountered during the iteration.
4. **Pointer Adjustment**: Move the pointer pointing to the shorter line inward. This is because moving the taller line would not increase the area (the height is constrained by the shorter line, and the width decreases), but moving the shorter line might lead to a taller li

In [25]:
prompt = """
Design and implement a Least Recently Used (LRU) Cache in Python. 
It should support two operations: get(key) and put(key, value).
Both operations must happen in O(1) time complexity.

Please provide the complete class implementation and explain which data structures you chose and why.
"""

result = ollama.generate(prompt=prompt)

print(result["text"])

To implement an LRU Cache with O(1) time complexity for both `get` and `put` operations, we use a combination of a **doubly linked list** and a **hash map (dictionary)**. The doubly linked list maintains the order of usage (most recently used at the front, least recently used at the end), while the hash map provides O(1) access to nodes in the linked list.

### Why These Data Structures?
- **Doubly Linked List**: 
  - Allows O(1) removal of the least recently used node (tail node) and O(1) insertion at the front (head).
  - Each node stores `key`, `value`, and pointers to `prev` and `next` nodes.
- **Hash Map (Dictionary)**:
  - Maps `key` to its corresponding node in the linked list for O(1) access.
  - Enables O(1) lookups during `get` and `put` operations.

### Implementation Details
1. **Dummy Head and Tail Nodes**:
   - Simplify edge cases (e.g., empty list) by adding dummy nodes at both ends of the linked list.
   - `head` points to the most recently used node, `tail` points to t

### Latency

In [26]:
print("Tokens:", result["total_tokens"])
print("Latency (s):", result["total_latency_sec"])
print("Tokens/sec:", result["gen_speed_tps"])

Tokens: 4563
Latency (s): 285.856
Tokens/sec: 15.78


## danielsheep/Qwen3-Coder-30B-A3B-Instruct-1M-Unsloth:UD-Q6_K_XL

In [30]:
import requests
import time


class OllamaClient:
    def __init__(
        self,
        model: str = "gpt-oss:20b",
        base_url: str = "http://localhost:11434",
        timeout: int = 300,
    ):
        self.model = model
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
    
    def generate(self, prompt: str) -> dict:
        url = f"{self.base_url}/api/generate"
        payload = {"model": self.model, "prompt": prompt, "stream": False}
    
        start_time = time.perf_counter()
        response = requests.post(url, json=payload)
        response.raise_for_status()
        total_wall_time = time.perf_counter() - start_time
        
        data = response.json()
    
        # Metrics from Ollama (in nanoseconds)
        prompt_tokens = data.get("prompt_eval_count", 0)
        output_tokens = data.get("eval_count", 0)
        
        # eval_duration is strictly the generation phase
        eval_duration_ns = data.get("eval_duration", 1) # Avoid div by zero
        
        # Calculate TPS based purely on generation time
        tokens_per_sec = output_tokens / (eval_duration_ns / 1e9)
    
        return {
            "text": data.get("response"),
            "input_tokens": prompt_tokens,
            "output_tokens": output_tokens,
            "total_tokens": prompt_tokens + output_tokens,
            "gen_speed_tps": round(tokens_per_sec, 2),
            "total_latency_sec": round(total_wall_time, 3)
        }


In [31]:
ollama = OllamaClient(model="danielsheep/Qwen3-Coder-30B-A3B-Instruct-1M-Unsloth:UD-Q6_K_XL")


### Reasoning

In [33]:
prompt = """
A bat and a ball cost $1.10 in total. 
The bat costs $1.00 more than the ball. 
How much does the ball cost? 
Think step-by-step and provide the final answer.
"""

result = ollama.generate(prompt=prompt)

print(result["text"])

Let me solve this step-by-step.

Let me define variables:
- Let x = cost of the ball (in dollars)
- Let y = cost of the bat (in dollars)

Now I'll set up equations based on the given information:

From the first statement: "A bat and a ball cost $1.10 in total"
x + y = 1.10

From the second statement: "The bat costs $1.00 more than the ball"
y = x + 1.00

Now I'll substitute the second equation into the first equation:
x + (x + 1.00) = 1.10

Simplifying:
x + x + 1.00 = 1.10
2x + 1.00 = 1.10
2x = 1.10 - 1.00
2x = 0.10
x = 0.05

So the ball costs $0.05 (5 cents).

Let me verify this answer:
- Ball costs: $0.05
- Bat costs: $0.05 + $1.00 = $1.05
- Total: $0.05 + $1.05 = $1.10 ✓
- The bat costs $1.00 more than the ball: $1.05 - $0.05 = $1.00 ✓

The ball costs 5 cents.


In [34]:
prompt = """
I put a diamond inside a glass cup. 
I place the glass cup inside a microwave. 
I turn the microwave on for 30 seconds. 
I take the glass cup out and put it in the freezer. 
Where is the diamond now and what is its state?
"""

result = ollama.generate(prompt=prompt)

print(result["text"])

The diamond is still inside the glass cup, and its state remains unchanged.

Here's what happens to each component:

**The diamond:**
- Remains intact and unchanged
- Diamonds are extremely durable and can withstand both microwave radiation and freezing temperatures
- The diamond's physical properties (hardness, structure) are unaffected by these conditions

**The glass cup:**
- May have experienced thermal stress from the rapid temperature change (microwave heating followed by freezing)
- Could potentially crack or shatter due to the temperature shock, but this depends on the glass type and construction

**The microwave:**
- The diamond doesn't cause any significant issues in the microwave
- It won't heat up much (diamonds are poor conductors of heat)
- No dangerous reactions occur

So the diamond is still safely inside the glass cup, but you should be cautious about the glass cup itself - it might be damaged from the thermal shock of going from microwave heat to freezer cold. If the 

### Logical

In [35]:
prompt = """
Evaluate the logical validity of this argument. Do not use real-world facts, only the provided statements.

Statement 1: All flurgs are green.
Statement 2: Some green things are electric.
Conclusion: Therefore, some flurgs are electric.

Is this conclusion logically valid? Explain why or why not.
"""

result = ollama.generate(prompt=prompt)

print(result["text"])

This argument is **logically invalid**.

Here's why:

**Statement 1: "All flurgs are green"** means: If something is a flurg → it is green
**Statement 2: "Some green things are electric"** means: There exists at least one thing that is both green and electric

The conclusion "Some flurgs are electric" would only follow if we could establish that at least one flurg is also electric.

However, the premises don't guarantee this. Here's a counterexample to show the invalidity:

- All flurgs are green (premise 1)
- Some green things are electric (premise 2)
- But it's possible that the electric things are all non-flurgs

For instance:
- Flurgs = {A, B, C} (all green)
- Electric things = {X, Y, Z} (all green, but none are flurgs)

In this case, both premises are satisfied, but the conclusion "some flurgs are electric" is false.

The logical structure is flawed because while we know all flurgs are green, and some green things are electric, this doesn't guarantee that any of the green things t

In [36]:
prompt = """
You are on an island where there are two types of people: Knights, who always tell the truth, and Knaves, who always lie.

You meet two people, A and B.
A says: "At least one of us is a Knave."

What are A and B? Walk through the logic to find the solution.
"""

result = ollama.generate(prompt=prompt)

print(result["text"])

I need to analyze A's statement "At least one of us is a Knave" to determine the types of A and B.

Let me consider both possibilities for A:

**Case 1: A is a Knight (tells truth)**
If A is a Knight, then A's statement "At least one of us is a Knave" must be true.
This means at least one of A or B is a Knave.
Since A is a Knight, B must be the Knave.

Let me verify: If A is a Knight and B is a Knave, then A's statement "At least one of us is a Knave" is true (B is the Knave), which is consistent with A being a Knight.

**Case 2: A is a Knave (lies)**
If A is a Knave, then A's statement "At least one of us is a Knave" must be false.
For this statement to be false, it must be that neither A nor B is a Knave.
This would mean both A and B are Knights.

But wait - if A is a Knave, then A cannot be a Knight. This creates a contradiction.

Let me re-examine Case 2 more carefully:
If A is a Knave, then A's statement is false.
The statement "At least one of us is a Knave" is false.
For this to

### Code

In [37]:
prompt = """
Write a Python function to solve the "Container With Most Water" problem. 
Given an array of integers representing heights of vertical lines, find two lines that together with the x-axis form a container, such that the container contains the most water.

Requirement: The solution must be O(n) time complexity. Explain your logic before writing the code.
"""

result = ollama.generate(prompt=prompt)

print(result["text"])

## Logic Explanation

This is a classic two-pointer problem that can be solved optimally using the **two-pointer technique**.

**Key Insights:**
1. The area of a container is determined by: `min(height[left], height[right]) * (right - left)`
2. To maximize area, we need to consider both height and width
3. **Greedy approach**: Start with the widest possible container (leftmost and rightmost lines)
4. **Strategy**: Move the pointer with smaller height inward, because:
   - The width will always decrease as we move pointers inward
   - If we move the taller line's pointer, we can only potentially decrease the area
   - If we move the shorter line's pointer, we might find a taller line that compensates for the reduced width

**Why this works:**
- We start with maximum width
- We always move the pointer that's limiting our height (the shorter line)
- This ensures we never miss the optimal solution
- Time complexity: O(n) - each element is visited at most once
- Space complexity: O(1)

## C

In [38]:
prompt = """
Design and implement a Least Recently Used (LRU) Cache in Python. 
It should support two operations: get(key) and put(key, value).
Both operations must happen in O(1) time complexity.

Please provide the complete class implementation and explain which data structures you chose and why.
"""

result = ollama.generate(prompt=prompt)

print(result["text"])

I'll design an LRU Cache with O(1) time complexity for both get and put operations using a combination of a hash map and a doubly linked list.

```python
class ListNode:
    """Doubly linked list node to store key-value pairs"""
    def __init__(self, key=0, value=0):
        self.key = key
        self.value = value
        self.prev = None
        self.next = None

class LRUCache:
    """
    LRU Cache implementation with O(1) time complexity for get and put operations.
    
    Data Structures Used:
    1. Hash Map (Dictionary): For O(1) key lookup
    2. Doubly Linked List: For maintaining access order and O(1) insertion/deletion
    
    The cache maintains a doubly linked list where:
    - Most recently used items are near the head
    - Least recently used items are near the tail
    - Hash map maps keys to their corresponding list nodes
    """
    
    def __init__(self, capacity: int):
        """
        Initialize the LRU Cache with given capacity.
        
        Args:
  

### Latency

In [40]:
print("Tokens:", result["total_tokens"])
print("Latency (s):", result["total_latency_sec"])
print("Tokens/sec:", result["gen_speed_tps"])

Tokens: 1797
Latency (s): 534.753
Tokens/sec: 3.39


## Score

In [1]:
from huggingface_hub import snapshot_download

local_dir = snapshot_download(
    repo_id="nomic-ai/nomic-embed-code",
    repo_type="model",        # optional, default is "model"
    cache_dir="hf_models",    # optional: where to store it
)Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

print("Model downloaded to:", local_dir)


/opt/homebrew/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 20 files: 100%|█████████████████████| 20/20 [1:00:48<00:00, 182.43s/it]

Model downloaded to: hf_models/models--nomic-ai--nomic-embed-code/snapshots/11114029805cee545ef111d5144b623787462a52


In [65]:
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder, util
import pandas as pd # Optional, if you want to save results to CSV later
import ast
import re
import textwrap

class ALLResponses:

    def GPT5_response(self):
        reasoning_prompt1 = r'''I can give the final answer with a brief explanation (without detailed step-by-step reasoning).
        
        Answer: The ball costs $0.05.
        
        Brief explanation:
        If the ball costs $0.05, then the bat costs $1.05 (which is $1.00 more). Together, $0.05 + $1.05 = $1.10, which matches the total cost.'''
        reasoning_prompt2 = r'''The diamond never goes anywhere.
        
        Where is it?
        It’s still inside the glass cup, and that cup is now in the freezer.
        
        What is its state?
        It’s still a solid diamond, essentially unchanged.
        
        A household microwave can’t melt or damage a diamond (it would require temperatures of thousands of degrees), and putting it in the freezer afterward doesn’t affect it either. At most, the diamond might have warmed slightly and then cooled again—but it remains a solid diamond the entire time.'''

        logic_prompt1 = r'''No, the conclusion is not logically valid.
        
        Why:
        
        Statement 1: All flurgs are green.
        → This means every flurg is included in the set of green things, but it does not say that any flurgs actually exist.
        
        Statement 2: Some green things are electric.
        → This says at least one green thing is electric, but it does not say that this green thing is a flurg.
        
        The conclusion — some flurgs are electric — would require a guaranteed overlap between flurgs and electric things. The premises do not establish that overlap.
        
        Logical counterexample (using only the given structure):
        
        All flurgs are green.
        
        Some green things are electric.
        
        But the electric green things could be non-flurgs.
        
        Therefore, it is possible that no flurgs are electric.
        
        Because the conclusion does not follow in all cases where the premises are true, the argument is logically invalid.'''
        logic_prompt2 = r'''Let’s analyze it step by step.
        The statement
        A says: “At least one of us is a Knave.”
        That means A is claiming that A is a Knave, or B is a Knave (or both).
        
        Case 1: Assume A is a Knight (always tells the truth)
        
        
        If A is telling the truth, then the statement must be true.
        
        
        So at least one of A or B is a Knave.
        
        
        But A is a Knight, so A is not a Knave.
        
        
        Therefore, B must be a Knave.
        
        
        This scenario is logically consistent.
        
        Case 2: Assume A is a Knave (always lies)
        
        
        If A is lying, then the statement must be false.
        
        
        The negation of “at least one of us is a Knave” is:
        
        “Neither of us is a Knave” (i.e., both are Knights).
        
        
        
        But that would mean A is a Knight, which contradicts the assumption that A is a Knave.
        
        
        This scenario is impossible.
        
        ✅ Conclusion
        The only consistent solution is:
        
        
        A is a Knight
        
        
        B is a Knave
        
        
        A truthfully states that at least one of them is a Knave, and that Knave is B.'''

        return reasoning_prompt1, reasoning_prompt2, logic_prompt1, logic_prompt2

    def Claude_code_logic_responses(self):
        code_prompt1 = r'''I'll explain the approach first, then provide the code.
        
        ## Logic
        
        The key insight is using a **two-pointer technique**:
        
        1. **Start with the widest container**: Place pointers at both ends of the array. The width is maximum here (right - left).
        
        2. **Calculate area at each step**: Area = width × min(height[left], height[right]). The minimum height determines how much water the container can hold.
        
        3. **Move the pointer with smaller height inward**: This is crucial. The width will only decrease as we move inward, so we can only improve the area by finding a taller line. Moving the taller pointer inward is pointless—it can only make the height smaller (limited by the smaller of the two), so the area shrinks.
        
        4. **Track the maximum area** found throughout the process.
        
        Why this is O(n): We iterate through the array exactly once. Each pointer moves from one end toward the center, visiting each element at most once.
        
        This greedy approach works because of a mathematical property: if we have two lines where left_height < right_height, moving the right pointer can never produce a better solution than moving the left pointer. So we safely skip those cases by always moving the shorter line.
        
        ```python
        def maxArea(heights):
            """
            Find the maximum area of water that can be contained.
            
            Args:
                heights: List of integers representing heights of vertical lines
                
            Returns:
                Integer representing the maximum area
                
            Time Complexity: O(n)
            Space Complexity: O(1)
            """
            max_area = 0
            left = 0
            right = len(heights) - 1
            
            while left < right:
                # Calculate current area
                width = right - left
                current_height = min(heights[left], heights[right])
                current_area = width * current_height
                
                # Update maximum area
                max_area = max(max_area, current_area)
                
                # Move the pointer pointing to the shorter line
                if heights[left] < heights[right]:
                    left += 1
                else:
                    right -= 1
            
            return max_area
        
        
        # Test cases
        if __name__ == "__main__":
            # Test 1: Example from problem
            heights1 = [1, 8, 6, 2, 5, 4, 8, 3, 7]
            print(f"Input: {heights1}")
            print(f"Output: {maxArea(heights1)}")  # Expected: 49 (between indices 1 and 8)
            print()
            
            # Test 2: Simple case
            heights2 = [1, 1]
            print(f"Input: {heights2}")
            print(f"Output: {maxArea(heights2)}")  # Expected: 1
            print()
            
            # Test 3: Increasing heights
            heights3 = [2, 3, 4, 5, 18, 17, 6]
            print(f"Input: {heights3}")
            print(f"Output: {maxArea(heights3)}")  # Expected: 17 (between indices 4 and 5)
            print()
            
            # Test 4: Single tall line
            heights4 = [1, 2, 4, 3]
            print(f"Input: {heights4}")
            print(f"Output: {maxArea(heights4)}")  # Expected: 4 (between indices 1 and 2)
        ```

        The function is straightforward in implementation but elegant in reasoning. The while loop runs at most n times (each element is visited once), making it optimal for this problem. The space complexity is O(1) since we only use a couple of variables regardless of input size.
        
        '''
        code_prompt2 = r'''```python
        from collections import OrderedDict
        
        class LRUCache:
            """
            Least Recently Used (LRU) Cache implementation with O(1) get and put operations.
            """
            
            def __init__(self, capacity: int):
                """
                Initialize the LRU cache.
                
                Args:
                    capacity: Maximum number of items the cache can hold
                """
                if capacity <= 0:
                    raise ValueError("Capacity must be positive")
                self.capacity = capacity
                self.cache = OrderedDict()
            
            def get(self, key: int) -> int:
                """
                Get value by key. Mark this key as recently used.
                
                Time Complexity: O(1)
                Space Complexity: O(1)
                
                Args:
                    key: The key to retrieve
                    
                Returns:
                    The value associated with the key, or -1 if not found
                """
                if key not in self.cache:
                    return -1
                
                # Move to end to mark as recently used
                self.cache.move_to_end(key)
                return self.cache[key]
            
            def put(self, key: int, value: int) -> None:
                """
                Add or update a key-value pair. Mark this key as recently used.
                If cache exceeds capacity, evict the least recently used item.
                
                Time Complexity: O(1)
                Space Complexity: O(1)
                
                Args:
                    key: The key to add or update
                    value: The value to store
                """
                # If key exists, remove it first so it gets re-added at the end
                if key in self.cache:
                    del self.cache[key]
                
                # Add/update the key-value pair (added to the end, most recently used)
                self.cache[key] = value
                
                # If we exceed capacity, remove the least recently used item (at the start)
                if len(self.cache) > self.capacity:
                    # popitem(last=False) removes the first item (least recently used)
                    self.cache.popitem(last=False)
            
            def __repr__(self) -> str:
                """String representation of cache contents (for debugging)."""
                return f"LRUCache(capacity={self.capacity}, items={dict(self.cache)})"
        
        
        # Example usage and test cases
        if __name__ == "__main__":
            # Test 1: Basic get and put
            cache = LRUCache(2)
            cache.put(1, 1)
            cache.put(2, 2)
            print(f"Test 1 - Get key 1: {cache.get(1)}")  # Output: 1
            
            cache.put(3, 3)  # Evicts key 2 (least recently used)
            print(f"Test 1 - Get key 2: {cache.get(2)}")  # Output: -1 (evicted)
            print(f"Test 1 - Cache state: {cache}\n")
            
            # Test 2: Accessing key updates recency
            cache = LRUCache(2)
            cache.put(1, 1)
            cache.put(2, 2)
            cache.get(1)  # Mark key 1 as recently used
            cache.put(3, 3)  # Evicts key 2, not key 1
            print(f"Test 2 - Get key 1: {cache.get(1)}")  # Output: 1
            print(f"Test 2 - Get key 2: {cache.get(2)}")  # Output: -1 (evicted)
            print(f"Test 2 - Cache state: {cache}\n")
            
            # Test 3: Updating existing key
            cache = LRUCache(2)
            cache.put(1, 1)
            cache.put(2, 2)
            cache.put(1, 10)  # Update key 1 (now most recently used)
            cache.put(3, 3)  # Evicts key 2
            print(f"Test 3 - Get key 1: {cache.get(1)}")  # Output: 10
            print(f"Test 3 - Get key 2: {cache.get(2)}")  # Output: -1 (evicted)
            print(f"Test 3 - Cache state: {cache}\n")
            
            # Test 4: Single item cache
            cache = LRUCache(1)
            cache.put(1, 1)
            print(f"Test 4 - Get key 1: {cache.get(1)}")  # Output: 1
            cache.put(2, 2)
            print(f"Test 4 - Get key 1: {cache.get(1)}")  # Output: -1 (evicted)
            print(f"Test 4 - Get key 2: {cache.get(2)}")  # Output: 2
            print(f"Test 4 - Cache state: {cache}")
            ```
            I've implemented an LRU Cache that achieves O(1) time complexity for both operations. Here's why this design works:
            
            ## Data Structure Choice: OrderedDict
            
            The key to O(1) performance is using **OrderedDict** from Python's collections module. OrderedDict maintains insertion order and provides O(1) access to all operations:
            
            - **get(key)**: Dictionary lookup is O(1), and `move_to_end()` is O(1)
            - **put(key, value)**: Dictionary operations are O(1), and `popitem(last=False)` is O(1)
            
            ## How It Works
            
            **Ordering Strategy**: Items are kept in order from least-recently-used (front) to most-recently-used (back).
            
            **get(key)**: 
            - Returns -1 if not found
            - If found, moves that key to the end (marking it as recently used)
            
            **put(key, value)**:
            - If the key already exists, delete it first (so it gets re-added at the end)
            - Add the key-value pair to the end (most recently used position)
            - If capacity is exceeded, remove the first item (least recently used)
            
            ## Why Not Use HashMap + Doubly Linked List?
            
            While the traditional approach uses a HashMap paired with a doubly-linked list, OrderedDict is more practical for Python because it:
            - Maintains the same O(1) time complexity
            - Simplifies the code significantly
            - Reduces potential for bugs from manual pointer management
            - Uses less memory overhead than managing explicit node objects
            
            The implementation includes comprehensive test cases demonstrating basic operations, recency updates, key updates, and edge cases.
            '''

        return code_prompt1, code_prompt2
        ifstat

    def gpt_oss20b(self):
        reasoning_prompt1 = r'''**Step‑by‑step solution**
        
        1. **Define the variables**  
           Let  
           \[
           \text{ball cost} = x \text{ dollars}
           \]  
           The bat is $1.00 more expensive than the ball, so  
           \[
           \text{bat cost} = x + 1.00 \text{ dollars}
           \]
        
        2. **Set up the total‑cost equation**  
           The problem says the bat and ball together cost $1.10.  
           \[
           x + (x + 1.00) = 1.10
           \]
        
        3. **Simplify the equation**  
           \[
           2x + 1.00 = 1.10
           \]
        
        4. **Solve for \(x\)**  
           Subtract 1.00 from both sides:  
           \[
           2x = 0.10
           \]  
           Divide by 2:  
           \[
           x = 0.05
           \]
        
        5. **Interpret the result**  
           \(x = 0.05\) dollars, which is **5 cents**.
        
        ---
        
        **Answer:** The ball costs **$0.05** (five cents).'''
        reasoning_prompt2 = r'''**Location:**  
        The diamond is still in the glass cup – now sitting in the freezer.
        
        **State:**  
        The diamond remains a solid, crystalline diamond.  
        It has not melted, sublimed, or changed phase; its structure is unchanged.  
        
        ---
        
        ### Why this is the case
        
        | Step | What happens (in short) | Effect on the diamond |
        |------|-------------------------|-----------------------|
        | 1.  Diamond in a glass cup | The diamond is physically inside the cup, protected from the microwave environment. | No direct interaction. |
        | 2.  Cup placed in microwave (30 s) | Microwaves heat polar molecules; glass (non‑polar but can absorb some microwave energy) may warm slightly, but the diamond is an excellent electrical insulator and doesn’t absorb microwave energy. | The diamond may receive a little heat by conduction from the cup, but it stays solid. |
        | 3.  Cup taken out & put in freezer | The cup and any absorbed heat cool. | The diamond cools back to ambient temperature. |
        
        **Key points**
        
        - **Microwave absorption:** Diamonds do not absorb microwaves; they are transparent to them. They cannot be heated this way.
        - **Glass safety:** A standard glass cup is microwave‑safe. 30 seconds is not enough to melt or shatter it.
        - **Phase stability:** Diamond has a very high melting point (~4,000 °C) and does not change phase at typical room or freezer temperatures. 
        
        Therefore, the diamond is still sitting inside the glass cup, now frozen at the freezer temperature, and remains a solid diamond.'''

        logic_prompt1 = r"""**No, the conclusion is not logically valid.**
        
        ---
        
        ### 1.  Translate the argument into predicate logic
        
        | English | Predicate logic |
        |---------|-----------------|
        | All flurgs are green. | ∀x (Flurg(x) → Green(x)) |
        | Some green things are electric. | ∃x (Green(x) ∧ Electric(x)) |
        | Therefore, some flurgs are electric. | ∃x (Flurg(x) ∧ Electric(x)) |
        
        The question is whether the premises entail the conclusion.
        
        ---
        
        ### 2.  Check the entailment
        
        To be valid, **every model** (interpretation) that makes the premises true must also make the conclusion true.  
        We can find a model in which the premises are true but the conclusion is false—this shows the argument is **invalid**.
        
        ---
        
        #### Counter‑model
        
        | Object | Flurg? | Green? | Electric? |
        |--------|--------|--------|-----------|
        | a      | No     | Yes    | Yes       |
        | b      | No     | Yes    | No        |
        | c      | No     | No     | No        |
        
        *Premise 1*  
        ∃x (Flurg(x) → Green(x)) is true for all x because there are no flurgs at all. A universal statement is vacuously true when its subject set is empty.
        
        *Premise 2*  
        There exists an x (object a) that is both green and electric, so ∃x (Green(x) ∧ Electric(x)) is true.
        
        *Conclusion*  
        There is **no** object that is both flurg and electric. Hence ∃x (Flurg(x) ∧ Electric(x)) is false.
        
        Thus we have a situation where the premises hold, but the conclusion fails. The argument is **not logically valid**.
        
        ---
        
        ### 3.  Intuitive explanation
        
        - **Premise 1** tells us every flurg (if any exist) is green.  
        - **Premise 2** tells us there is at least one green object that is electric, but it says nothing about that object’s membership in the “flurg” class.
        
        From these two facts we cannot deduce that *a* flurg (or any flurg) is electric. It could be that the electric green object is a non‑flurg, and no flurg is electric at all.
        
        ---
        
        ### 4.  What would make it valid?
        
        If we added an extra premise that **some** flurg is electric *or* that **every** green is flurg, then the conclusion could be derived. As it stands, however, the inference fails.
        
        ---
        
        **Conclusion:** The argument is *invalid*; the premises do not logically entail the conclusion."""
        logic_prompt2 = r"""Let’s translate the situation into a short logical form.
        
        ---
        
        ### 1.  Translate the sentence
        
        > “At least one of us is a Knave.”
        
        Let  
        - \(A\) = “A is a Knave”  
        - \(B\) = “B is a Knave”
        
        The sentence is simply  
        
        \[
        S \;=\; (A \text{ or } B)
        \]
        
        So \(S\) is true exactly when **at least one** of the two people is a Knave.
        
        ---
        
        ### 2.  Apply the Knights‑and‑Knaves rule
        
        - If a person is a **Knight**, the statement he says must be **true**.  
        - If a person is a **Knave**, the statement he says must be **false**.
        
        We only have one statement: A says \(S\).  
        Therefore:
        
        | A’s type | What \(S\) must be | What this implies for A and B |
        |----------|-------------------|--------------------------------|
        | Knight   | \(S\) **true**    | “At least one of us is a Knave” is true. A cannot be a Knave, so B must be a Knave. |
        | Knave    | \(S\) **false**   | “At least one of us is a Knave” is false → **neither** is a Knave. That means both A and B are Knights, contradicting that A is a Knave. |
        
        The second row gives a contradiction, so **A cannot be a Knave**.
        
        ---
        
        ### 3.  The only consistent assignment
        
        - A is a **Knight**.  
        - For \(S\) to be true, B must be a **Knave**.
        
        Check:  
        - A (Knight) says “At least one of us is a Knave”.  
        - B is indeed a Knave, so the statement is true.  
        - No contradictions appear.
        
        ---
        
        ## Answer
        
        **A is a Knight and B is a Knave.**  
        The logic follows from the fact that a Knight’s statement must be true, a Knave’s statement must be false, and the sentence “At least one of us is a Knave” forces the presence of a Knave if A is telling the truth. A being a Knave would make the sentence true, which is impossible for a liar. Thus the only solution is A = Knight, B = Knave."""

        code_prompt1 = r'''**Explanation – Why a Two‑Pointer Scan Works**
        
        The goal is to pick two indices `i` and `j` (`i < j`) that maximize the area
        
        ```
        area(i, j) = min(height[i], height[j]) * (j – i)
        ```
        
        A brute‑force double loop would take `O(n²)` time.  
        We can do better because:
        
        1. **The limiting factor is the shorter line**  
           For any fixed pair `(i, j)` the height of the water that can be held is bounded by the shorter of the two lines.  
           So if we keep the shorter side and try to find a *taller* line further away, we might increase the area.
        
        2. **Moving the longer side never helps**  
           Suppose `height[i] < height[j]`.  
           If we move the right pointer `j` leftwards (towards `i`), the width `(j – i)` decreases, and the height cannot increase beyond `height[i]` (the shorter side).  
           Therefore any new pair that keeps the left pointer `i` fixed will have an area no larger than the current one.  
           Consequently we **must move the pointer that is on the shorter line**.
        
        By maintaining two indices at the ends of the array and iteratively moving the shorter one inward, we examine all potentially optimal pairs in a single pass.
        
        **Algorithm Outline**
        
        ```
        left  = 0
        right = n-1
        max_area = 0
        
        while left < right:
            width = right - left
            height = min(height[left], height[right])
            max_area = max(max_area, width * height)
        
            # Move the pointer at the shorter line
            if height[left] < height[right]:
                left += 1
            else:
                right -= 1
        
        return max_area
        ```
        
        - Each pointer moves at most `n` times → **O(n)** total operations.
        - Only a few integer variables → **O(1)** extra space.
        
        ---
        
        ## Python Implementation
        
        ```python
        from typing import List
        
        def max_area(height: List[int]) -> int:
            """
            Find the maximum amount of water a container can hold.
        
            Parameters
            ----------
            height : List[int]
                A list of non‑negative integers representing the heights of vertical lines
                drawn on the x‑axis at integer positions.
        
            Returns
            -------
            int
                The maximum area of water that can be contained.
        
            Complexity
            ----------
            Time: O(n), where n is len(height).
            Space: O(1).
            """
            left, right = 0, len(height) - 1
            max_area = 0
        
            while left < right:
                # Current width and limiting height
                width = right - left
                h = height[left] if height[left] < height[right] else height[right]
        
                # Update the best area found so far
                area = width * h
                if area > max_area:
                    max_area = area
        
                # Move the pointer on the shorter side
                if height[left] < height[right]:
                    left += 1
                else:
                    right -= 1
        
            return max_area
        ```
        
        ### Quick Test
        
        ```python
        if __name__ == "__main__":
            examples = [
                ([1,8,6,2,5,4,8,3,7], 49),
                ([1,1], 1),
                ([4,3,2,1,4], 16),
                ([1,2,1], 2),
            ]
        
            for arr, expected in examples:
                result = max_area(arr)
                assert result == expected, f"Failed for {arr}: expected {expected}, got {result}"
            print("All tests passed!")
        ```
        
        The function follows the two‑pointer strategy described above, guaranteeing linear time complexity while using constant additional memory.'''
        code_prompt2 = r'''**Solution Overview**
        
        An LRU (Least‑Recently‑Used) cache keeps the most recently accessed items in memory while evicting the oldest items once the cache reaches its capacity.  
        The two required operations are
        
        | Operation | Complexity target | What it must do |
        |-----------|-------------------|-----------------|
        | `get(key)` | O(1) | Return the value for *key* (or `-1`/`None` if missing) **and** mark the key as most‑recently used |
        | `put(key, value)` | O(1) | Insert or update the value for *key* and mark it as most‑recently used. If the cache is full, evict the least‑recently used item |
        
        To achieve **O(1)** for both operations we need
        
        1. **Constant‑time lookup of a key → value**  
           → a hash table (`dict` in Python).
        
        2. **Constant‑time update of “recency” order**  
           → a data structure that lets us remove a node from the middle of a list and move it to the front in O(1).  
           → a *doubly‑linked list* is perfect for this because each node knows its previous and next node, so we can unlink it in constant time.
        
        By keeping the *most‑recently used* element at the **head** of the list and the *least‑recently used* element at the **tail**, we can:
        
        * **Eviction** – pop the tail node when capacity is exceeded (O(1)).
        * **Promotion** – move a node to the head when it is accessed or inserted (O(1)).
        
        The hash table stores a mapping `key → node`, where `node` is the node in the doubly‑linked list.  
        Thus, every operation touches only the dictionary (hash look‑up) and a constant number of pointer updates in the list – both O(1).
        
        ---
        
        ## Implementation
        
        Below is a complete, self‑contained Python class that follows the design above.
        
        ```python
        class LRUCache:
            """
            A Least‑Recently‑Used (LRU) cache that supports O(1) get and put operations.
        
            Attributes
            ----------
            capacity : int
                Maximum number of key/value pairs that can be stored.
            cache : dict
                Maps keys to the corresponding node in the linked list.
            head : Node
                Dummy head of the doubly‑linked list (most recent item is head.next).
            tail : Node
                Dummy tail of the doubly‑linked list (least recent item is tail.prev).
            """
        
            class _Node:
                """Internal node class for the doubly‑linked list."""
                __slots__ = ("key", "value", "prev", "next")
        
                def __init__(self, key=None, value=None):
                    self.key = key
                    self.value = value
                    self.prev = None
                    self.next = None
        
            def __init__(self, capacity: int):
                if capacity <= 0:
                    raise ValueError("Capacity must be a positive integer.")
                self.capacity = capacity
                self.cache = {}          # key → node
                # Create dummy head and tail to avoid edge‑case checks
                self.head = self._Node()
                self.tail = self._Node()
                self.head.next = self.tail
                self.tail.prev = self.head
        
            # ----------------- Internal helpers -----------------
            def _remove(self, node: _Node) -> None:
                """Unlink *node* from the doubly‑linked list."""
                prev_node = node.prev
                next_node = node.next
                prev_node.next = next_node
                next_node.prev = prev_node
        
            def _add_to_front(self, node: _Node) -> None:
                """Insert *node* right after the dummy head (most recent position)."""
                first = self.head.next
                self.head.next = node
                node.prev = self.head
                node.next = first
                first.prev = node
        
            def _move_to_front(self, node: _Node) -> None:
                """Move an existing node to the front (most recent)."""
                self._remove(node)
                self._add_to_front(node)
        
            def _evict_lru(self) -> None:
                """
                Remove the least‑recently used node (node just before the dummy tail)
                and delete its key from the cache dictionary.
                """
                lru_node = self.tail.prev
                if lru_node is self.head:      # Cache is empty – should not happen
                    return
                self._remove(lru_node)
                del self.cache[lru_node.key]
        
            # ----------------- Public API -----------------
            def get(self, key: int) -> int:
                """Return the value for *key* if present, else -1.
                Mark the key as most recently used."""
                node = self.cache.get(key)
                if node is None:
                    return -1
                # Promote node to most recent position
                self._move_to_front(node)
                return node.value
        
            def put(self, key: int, value: int) -> None:
                """Insert or update the key/value pair.
                If capacity is exceeded, evict the least‑recently used item."""
                node = self.cache.get(key)
        
                if node:
                    # Update value and move to front
                    node.value = value
                    self._move_to_front(node)
                else:
                    # New key
                    if len(self.cache) >= self.capacity:
                        self._evict_lru()
                    new_node = self._Node(key, value)
                    self.cache[key] = new_node
                    self._add_to_front(new_node)
        
            # Optional helper for debugging / testing
            def __repr__(self) -> str:
                vals = []
                curr = self.head.next
                while curr != self.tail:
                    vals.append(f"{curr.key}:{curr.value}")
                    curr = curr.next
                return "LRUCache(" + " <-> ".join(vals) + ")"
        ```
        
        ### How it works
        
        | Operation | Steps performed | Time complexity |
        |-----------|-----------------|-----------------|
        | **`get(key)`** | 1. Look up `key` in the dictionary. <br>2. If present, move the node to the front of the list. <br>3. Return the value. | O(1) (hash lookup + O(1) pointer ops) |
        | **`put(key, value)`** | 1. Check if key already exists. <br>2. If it does, update value and move node to front. <br>3. If it doesn’t: <br> • If cache is full, evict tail‑prev node. <br> • Create new node, insert it at front, and add to dictionary. | O(1) (all operations are constant‑time) |
        
        ---
        
        ## Why these data structures?
        
        | Data structure | Reason for selection |
        |----------------|----------------------|
        | **Dictionary (`dict`)** | O(1) average‑case key lookup and update. Stores *key → node* so we can instantly access the node that represents that key. |
        | **Doubly‑Linked List** | Allows O(1) removal and insertion of any node when we already have a reference to it. By keeping dummy head/tail nodes we avoid special‑case logic for empty or single‑node lists. |
        | **Node with `__slots__`** | Keeps each node lightweight (no `__dict__`) which is handy when the cache holds thousands of items. |
        
        ---
        
        ## Example usage
        
        ```python
        if __name__ == "__main__":
            cache = LRUCache(3)
        
            cache.put(1, 10)
            cache.put(2, 20)
            cache.put(3, 30)
            print(cache)          # LRUCache(3:30 <-> 2:20 <-> 1:10)
        
            print(cache.get(2))   # 20
            print(cache)          # LRUCache(2:20 <-> 3:30 <-> 1:10)
        
            cache.put(4, 40)      # Evicts key 1 (least recent)
            print(cache)          # LRUCache(4:40 <-> 2:20 <-> 3:30)
        
            print(cache.get(1))   # -1 (not found)
        ```
        
        The example demonstrates that:
        
        * The most recently accessed items bubble to the front.  
        * When the capacity is exceeded, the tail‑node (least recently used) is evicted.  
        * All operations happen in constant time, satisfying the problem requirement.'''
        
        return reasoning_prompt1, reasoning_prompt2, logic_prompt1, logic_prompt2, code_prompt1, code_prompt2

    def qwen330b(self):
        reasoning_prompt1 = r"""To solve the problem, we begin by defining the cost of the ball as $ x $ dollars. According to the problem:
        
        - The **bat** costs **$1.00 more** than the ball.
        - Therefore, the **bat costs** $ x + 1.00 $.
        - The **total cost** of both the bat and the ball is **$1.10**.
        
        ---
        
        ### Step 1: Set up the equation
        
        $$
        \text{Cost of ball} + \text{Cost of bat} = 1.10
        $$
        $$
        x + (x + 1.00) = 1.10
        $$
        
        ---
        
        ### Step 2: Simplify the equation
        
        $$
        x + x + 1.00 = 1.10
        $$
        $$
        2x + 1.00 = 1.10
        $$
        
        ---
        
        ### Step 3: Solve for $ x $
        
        Subtract 1.00 from both sides:
        
        $$
        2x = 0.10
        $$
        
        Divide both sides by 2:
        
        $$
        x = 0.05
        $$
        
        ---
        
        ### Step 4: Interpret the result
        
        - The **ball costs $0.05** (or 5 cents).
        - The **bat costs $0.05 + $1.00 = $1.05**.
        - Total: $0.05 + $1.05 = $1.10 ✅
        
        ---
        
        ### Common Mistake to Avoid
        
        Many people intuitively think the ball costs $0.10, which would make the bat cost $1.10, leading to a total of $1.20 — **not** $1.10. This is why it's important to set up the equation properly instead of relying on intuition.
        
        ---
        
        ### Final Answer:
        
        $$
        \boxed{0.05}
        $$"""
        reasoning_prompt2 = r"""Based on the properties of diamonds, glass, microwaves, and freezing, here's a clear breakdown of what happens:
        
        ### 1. **Microwave Step (30 seconds):**
           - **Diamond behavior:** Diamonds are **not affected by microwave radiation**. Microwaves heat materials by exciting *water molecules* or *polar substances*, but diamonds (pure carbon) are non-polar and do not absorb microwave energy. The diamond **remains unchanged**—it does not heat up, melt, or react.
           - **Glass cup behavior:** Standard glass (e.g., borosilicate or soda-lime) is microwave-safe and **does not heat significantly** in a short 30-second cycle. It may feel slightly warm from ambient air, but it won't become hot enough to shatter or alter the diamond.
           - **Result:** The diamond stays intact inside the glass cup. No damage occurs.
        
        ### 2. **Freezer Step:**
           - **Diamond behavior:** Diamonds are **extremely stable** across a wide temperature range. Freezing (typically -18°C / 0°F) has **no effect** on a diamond’s structure, hardness, or state. It remains solid, lustrous, and chemically unchanged.
           - **Glass cup behavior:** Glass contracts slightly when cooled but **does not crack** under normal freezer conditions (assuming the cup was at room temperature after microwaving, not hot). No risk of shattering.
           - **Result:** The diamond remains inside the glass cup, now cold but physically identical to before.
        
        ### Final Answer:
        - **Where is the diamond?**  
          **Still inside the glass cup**, unharmed and undisturbed.
          
        - **What is its state?**  
          **Solid, unchanged, and intact**—exactly as it was before the experiment.  
          *(Diamonds have a melting point of ~4,000°C / 7,232°F, far above microwave or freezer temperatures. Freezing cannot alter its crystalline structure.)*
        
        ### Why This Isn’t a Myth (Common Misconceptions Debunked):
        - ❌ *"Microwaves melt diamonds!"* → **False.** Microwaves don’t heat diamonds (they don’t absorb microwaves), and even if they did, microwaves can’t reach diamond-melting temperatures.
        - ❌ *"The glass will shatter in the freezer!"* → **False.** The cup wasn’t hot after microwaving (only ambient temperature), so no thermal shock occurs.
        - ❌ *"Diamonds react with cold!"* → **False.** Diamonds are inert to temperature changes within this range.
        
        ### Practical Note:
        This experiment is **completely safe** for the diamond. However, *never* put metal objects (e.g., gold jewelry with metal settings) or non-microwave-safe glass in a microwave—**but diamonds are safe**. The cup might feel slightly cool from the freezer, but the diamond remains untouched. 
        
        > 💎 **In short: The diamond is still in the cup, as solid and pristine as when you started.**"""
        
        logic_prompt1 = r"""The argument is **logically invalid**. Here's why:
        
        1. **Statement 1** ("All flurgs are green") means the set of flurgs is entirely contained within the set of green things.  
        2. **Statement 2** ("Some green things are electric") means there exists *at least one* green thing that is electric, but it does **not** specify that this electric green thing is a flurg.  
        3. The conclusion ("Some flurgs are electric") requires that **at least one flurg** is electric. However, the premises do not guarantee this. The "some green things" that are electric could be **non-flurgs** (e.g., green electric cars), while all flurgs might still be non-electric.  
        
        **Example to illustrate**:  
        - Let "flurgs" = green apples (all green).  
        - Let "some green things that are electric" = green electric cars (not apples).  
        Here, all flurgs (apples) are green, and some green things (cars) are electric, but **no flurgs are electric**. The premises are true, but the conclusion is false.  
        
        Since the conclusion does not **necessarily** follow from the premises (it could be false even if the premises are true), the argument is **invalid**. The error is a classic **fallacy of the undistributed middle**—the middle term ("green") is not distributed in either premise, so no valid conclusion about the overlap between flurgs and electric things can be drawn.  
        
        **Answer**: The conclusion is **not logically valid**."""
        logic_prompt2 = r"""To determine the identities of A and B, we analyze the logical implications of A's statement: **"At least one of us is a Knave."**
        
        ---
        
        ### Step 1: Understand the Types
        
        - **Knights** always tell the truth.
        - **Knaves** always lie.
        
        So, if A says something, it must be **true** if A is a Knight, and **false** if A is a Knave.
        
        ---
        
        ### Step 2: Analyze A's Statement
        
        A says: **"At least one of us is a Knave."**
        
        This is logically equivalent to saying:  
        > **Either A is a Knave, or B is a Knave (or both).**
        
        The **negation** of this statement (i.e., what would be true if A were lying) is:  
        > **Neither A nor B is a Knave**, i.e., **both are Knights**.
        
        ---
        
        ### Step 3: Consider All Possibilities
        
        Let’s test all four possible combinations for A and B:
        
        | A      | B      | Is A's statement true? | A's identity consistent? |
        |--------|--------|------------------------|---------------------------|
        | Knight | Knight | No (both are Knights)  | ❌ A would be lying → contradiction |
        | Knight | Knave  | Yes (B is a Knave)     | ✅ A tells the truth → consistent |
        | Knave  | Knight | No (A lies, so the truth is both Knights) → but A is a Knave → contradiction | ❌ |
        | Knave  | Knave  | Yes (both are Knaves)  | ❌ A lies, but the statement is true → contradiction |
        
        Only **one case** is logically consistent:
        
        - **A is a Knight**, and **B is a Knave**.
        
        ---
        
        ### Step 4: Final Verification
        
        - A is a **Knight**, so A tells the truth.
        - A says: "At least one of us is a Knave."
        - Since A is a Knight, the statement must be **true**.
        - Therefore, **B must be a Knave** (since A is not a Knave).
        - This matches the scenario where **A = Knight, B = Knave**.
        
        All other scenarios lead to contradictions.
        
        ---
        
        ### ✅ Final Answer:
        
        **A is a Knight, and B is a Knave.**
        
"""

        code_prompt1 = r"""To solve the "Container With Most Water" problem efficiently with O(n) time complexity, we use a two-pointer technique that systematically explores potential containers while maximizing the area.
        
        ### Approach
        1. **Initialization**: Start with two pointers, `left` at the beginning (index 0) and `right` at the end (index `len(heights) - 1`) of the array. This represents the widest possible container.
        2. **Area Calculation**: For each pair of lines pointed to by `left` and `right`, calculate the area as the product of the distance between them (`right - left`) and the height of the shorter line (`min(heights[left], heights[right])`).
        3. **Update Maximum Area**: Track the maximum area encountered during the iteration.
        4. **Pointer Adjustment**: Move the pointer pointing to the shorter line inward. This is because moving the taller line would not increase the area (the height is constrained by the shorter line, and the width decreases), but moving the shorter line might lead to a taller line, potentially increasing the area despite the reduced width.
        5. **Termination**: Continue until the `left` and `right` pointers meet, ensuring all possible containers are evaluated in O(n) time.
        
        ### Solution Code
        ```python
        def maxArea(heights):
            left = 0
            right = len(heights) - 1
            max_area = 0
            while left < right:
                width = right - left
                h = min(heights[left], heights[right])
                current_area = h * width
                if current_area > max_area:
                    max_area = current_area
                if heights[left] < heights[right]:
                    left += 1
                else:
                    right -= 1
            return max_area
        ```
        
        ### Explanation
        - **Initialization**: The `left` pointer starts at the first element, and the `right` pointer starts at the last element, forming the widest possible container.
        - **Loop**: While `left` is less than `right`, compute the area formed by the current container. Update `max_area` if the current area is larger.
        - **Pointer Movement**: If the height at `left` is less than the height at `right`, increment `left` to potentially find a taller line. Otherwise, decrement `right`. This ensures that we always explore containers that could potentially hold more water.
        - **Efficiency**: Each iteration moves one pointer closer to the other, resulting in exactly `n-1` iterations (where `n` is the number of elements), leading to O(n) time complexity. The space complexity is O(1) as only a few variables are used.
        
        This approach efficiently narrows down the search space by leveraging the two-pointer technique, ensuring optimal performance without unnecessary computations."""
        code_prompt2 = r"""To implement an LRU Cache with O(1) time complexity for both `get` and `put` operations, we use a combination of a **doubly linked list** and a **hash map (dictionary)**. The doubly linked list maintains the order of usage (most recently used at the front, least recently used at the end), while the hash map provides O(1) access to nodes in the linked list.
        
        ### Why These Data Structures?
        - **Doubly Linked List**: 
          - Allows O(1) removal of the least recently used node (tail node) and O(1) insertion at the front (head).
          - Each node stores `key`, `value`, and pointers to `prev` and `next` nodes.
        - **Hash Map (Dictionary)**:
          - Maps `key` to its corresponding node in the linked list for O(1) access.
          - Enables O(1) lookups during `get` and `put` operations.
        
        ### Implementation Details
        1. **Dummy Head and Tail Nodes**:
           - Simplify edge cases (e.g., empty list) by adding dummy nodes at both ends of the linked list.
           - `head` points to the most recently used node, `tail` points to the least recently used node.
        
        2. **`get(key)`**:
           - If `key` is in the cache, move its node to the front (head) and return its value.
           - If `key` is not found, return `-1`.
        
        3. **`put(key, value)`**:
           - If `key` exists, update its value, move the node to the front, and return.
           - If `key` does not exist:
             - If the cache is full, remove the least recently used node (tail's previous node) from both the linked list and the cache.
             - Add the new node to the front of the linked list and store it in the cache.
        
        ### Complete Implementation
        ```python
        class LRUCache:
            class Node:
                __slots__ = ('key', 'value', 'prev', 'next')
                def __init__(self, key, value):
                    self.key = key
                    self.value = value
                    self.prev = None
                    self.next = None
        
            def __init__(self, capacity: int):
                self.capacity = capacity
                self.cache = {}
                # Initialize dummy head and tail
                self.head = self.Node(None, None)
                self.tail = self.Node(None, None)
                self.head.next = self.tail
                self.tail.prev = self.head
        
            def get(self, key: int) -> int:
                if key in self.cache:
                    node = self.cache[key]
                    self.remove_node(node)
                    self.add_to_head(node)
                    return node.value
                return -1
        
            def put(self, key: int, value: int) -> None:
                if key in self.cache:
                    node = self.cache[key]
                    node.value = value
                    self.remove_node(node)
                    self.add_to_head(node)
                else:
                    if len(self.cache) == self.capacity:
                        self.remove_node(self.tail.prev)
                        del self.cache[self.tail.prev.key]
                    new_node = self.Node(key, value)
                    self.add_to_head(new_node)
                    self.cache[key] = new_node
        
            def remove_node(self, node):
                node.prev.next = node.next
                node.next.prev = node.prev
        
            def add_to_head(self, node):
                node.next = self.head.next
                node.prev = self.head
                self.head.next.prev = node
                self.head.next = node
        ```
        
        ### Explanation
        - **Initialization**:
          - `head` and `tail` are dummy nodes forming the boundaries of the linked list.
          - `cache` is a dictionary mapping keys to nodes.
        
        - **`get(key)`**:
          - Checks if `key` exists in `cache`.
          - If found, moves the node to the front (most recently used) via `remove_node` and `add_to_head`.
          - Returns the node's value or `-1` if not found.
        
        - **`put(key, value)`**:
          - **Update existing key**: 
            - Updates the node's value, moves it to the front.
          - **New key**:
            - If cache is full, removes the least recently used node (tail's previous node) from both the list and `cache`.
            - Creates a new node, adds it to the front, and stores it in `cache`.
        
        - **Helper Methods**:
          - `remove_node(node)`: Removes a node from the linked list in O(1) time.
          - `add_to_head(node)`: Adds a node to the front of the linked list in O(1) time.
        
        ### Time Complexity
        - **`get`**: O(1) (dictionary lookup + O(1) list operations).
        - **`put`**: O(1) (dictionary lookup/update + O(1) list operations).
        
        This implementation efficiently maintains the LRU property while ensuring optimal performance for both operations. The use of dummy nodes simplifies edge-case handling, and the hash map ensures constant-time access to nodes."""

        return reasoning_prompt1, reasoning_prompt2, logic_prompt1, logic_prompt2, code_prompt1, code_prompt2

    def Danielsheep_Qwen3_Coder(self):
        reasoning_prompt1 = r"""Let me solve this step-by-step.
        
        Let me define variables:
        - Let x = cost of the ball (in dollars)
        - Let y = cost of the bat (in dollars)
        
        Now I'll set up equations based on the given information:
        
        From the first statement: "A bat and a ball cost $1.10 in total"
        x + y = 1.10
        
        From the second statement: "The bat costs $1.00 more than the ball"
        y = x + 1.00
        
        Now I'll substitute the second equation into the first equation:
        x + (x + 1.00) = 1.10
        
        Simplifying:
        x + x + 1.00 = 1.10
        2x + 1.00 = 1.10
        2x = 1.10 - 1.00
        2x = 0.10
        x = 0.05
        
        So the ball costs $0.05 (5 cents).
        
        Let me verify this answer:
        - Ball costs: $0.05
        - Bat costs: $0.05 + $1.00 = $1.05
        - Total: $0.05 + $1.05 = $1.10 ✓
        - The bat costs $1.00 more than the ball: $1.05 - $0.05 = $1.00 ✓
        
        The ball costs 5 cents."""
        reasoning_prompt2 = r"""The diamond is still inside the glass cup, and its state remains unchanged.
        
        Here's what happens to each component:
        
        **The diamond:**
        - Remains intact and unchanged
        - Diamonds are extremely durable and can withstand both microwave radiation and freezing temperatures
        - The diamond's physical properties (hardness, structure) are unaffected by these conditions
        
        **The glass cup:**
        - May have experienced thermal stress from the rapid temperature change (microwave heating followed by freezing)
        - Could potentially crack or shatter due to the temperature shock, but this depends on the glass type and construction
        
        **The microwave:**
        - The diamond doesn't cause any significant issues in the microwave
        - It won't heat up much (diamonds are poor conductors of heat)
        - No dangerous reactions occur
        
        So the diamond is still safely inside the glass cup, but you should be cautious about the glass cup itself - it might be damaged from the thermal shock of going from microwave heat to freezer cold. If the glass is fragile, it could break and potentially cause the diamond to fall out.
        
        The diamond itself remains in its original state - a hard, durable crystal structure."""
        
        logic_prompt1 = r"""This argument is **logically invalid**.
        
        Here's why:
        
        **Statement 1: "All flurgs are green"** means: If something is a flurg → it is green
        **Statement 2: "Some green things are electric"** means: There exists at least one thing that is both green and electric
        
        The conclusion "Some flurgs are electric" would only follow if we could establish that at least one flurg is also electric.
        
        However, the premises don't guarantee this. Here's a counterexample to show the invalidity:
        
        - All flurgs are green (premise 1)
        - Some green things are electric (premise 2)
        - But it's possible that the electric things are all non-flurgs
        
        For instance:
        - Flurgs = {A, B, C} (all green)
        - Electric things = {X, Y, Z} (all green, but none are flurgs)
        
        In this case, both premises are satisfied, but the conclusion "some flurgs are electric" is false.
        
        The logical structure is flawed because while we know all flurgs are green, and some green things are electric, this doesn't guarantee that any of the green things that are electric are also flurgs. The electric green things could be entirely separate from the flurgs.
        
        The argument commits the logical fallacy of assuming that because A is a subset of B, and some B's have property C, then some A's must have property C. This is not necessarily true."""
        logic_prompt2 = r"""I need to analyze A's statement "At least one of us is a Knave" to determine the types of A and B.
        
        Let me consider both possibilities for A:
        
        **Case 1: A is a Knight (tells truth)**
        If A is a Knight, then A's statement "At least one of us is a Knave" must be true.
        This means at least one of A or B is a Knave.
        Since A is a Knight, B must be the Knave.
        
        Let me verify: If A is a Knight and B is a Knave, then A's statement "At least one of us is a Knave" is true (B is the Knave), which is consistent with A being a Knight.
        
        **Case 2: A is a Knave (lies)**
        If A is a Knave, then A's statement "At least one of us is a Knave" must be false.
        For this statement to be false, it must be that neither A nor B is a Knave.
        This would mean both A and B are Knights.
        
        But wait - if A is a Knave, then A cannot be a Knight. This creates a contradiction.
        
        Let me re-examine Case 2 more carefully:
        If A is a Knave, then A's statement is false.
        The statement "At least one of us is a Knave" is false.
        For this to be false, the negation must be true: "Neither A nor B is a Knave."
        This means both A and B are Knights.
        But this contradicts our assumption that A is a Knave.
        
        Therefore, Case 2 is impossible.
        
        **Conclusion:**
        A must be a Knight, and B must be a Knave.
        
        Let me double-check:
        - A (Knight) says "At least one of us is a Knave" - this is true because B is a Knave ✓
        - B (Knave) would lie about this statement, but we don't know what B says since we only have A's statement ✓
        
        Therefore: **A is a Knight and B is a Knave.**"""

        code_prompt1 = r'''## Logic Explanation
        
        This is a classic two-pointer problem that can be solved optimally using the **two-pointer technique**.
        
        **Key Insights:**
        1. The area of a container is determined by: `min(height[left], height[right]) * (right - left)`
        2. To maximize area, we need to consider both height and width
        3. **Greedy approach**: Start with the widest possible container (leftmost and rightmost lines)
        4. **Strategy**: Move the pointer with smaller height inward, because:
           - The width will always decrease as we move pointers inward
           - If we move the taller line's pointer, we can only potentially decrease the area
           - If we move the shorter line's pointer, we might find a taller line that compensates for the reduced width
        
        **Why this works:**
        - We start with maximum width
        - We always move the pointer that's limiting our height (the shorter line)
        - This ensures we never miss the optimal solution
        - Time complexity: O(n) - each element is visited at most once
        - Space complexity: O(1)
        
        ## Code Implementation
        
        ```python
        def max_area(height):
            """
            Find the maximum area of water that can be stored between two lines.
            
            Args:
                height: List of integers representing heights of vertical lines
                
            Returns:
                int: Maximum area of water that can be stored
                
            Time Complexity: O(n)
            Space Complexity: O(1)
            """
            # Initialize two pointers at the start and end
            left = 0
            right = len(height) - 1
            max_water = 0
            
            # Continue while pointers haven't crossed
            while left < right:
                # Calculate current area
                current_area = min(height[left], height[right]) * (right - left)
                max_water = max(max_water, current_area)
                
                # Move the pointer with smaller height inward
                if height[left] < height[right]:
                    left += 1
                else:
                    right -= 1
            
            return max_water
        
        # Test cases
        def test_max_area():
            # Test case 1: Example from problem
            assert max_area([1,8,6,2,5,4,8,3,7]) == 49
            
            # Test case 2: Simple case
            assert max_area([1,1]) == 1
            
            # Test case 3: Increasing heights
            assert max_area([1,2,3,4,5]) == 6
            
            # Test case 4: Decreasing heights
            assert max_area([5,4,3,2,1]) == 6
            
            # Test case 5: Mixed case
            assert max_area([2,3,4,5,18,17,6]) == 17
            
            print("All test cases passed!")
        
        # Run tests
        test_max_area()
        
        # Example walkthrough
        def max_area_with_trace(height):
            """
            Same function but with trace to show the process
            """
            left = 0
            right = len(height) - 1
            max_water = 0
            step = 0
            
            print(f"Array: {height}")
            print("Step by step process:")
            
            while left < right:
                step += 1
                current_area = min(height[left], height[right]) * (right - left)
                max_water = max(max_water, current_area)
                
                print(f"Step {step}: left={left}, right={right}, "
                      f"height[{left}]={height[left]}, height[{right}]={height[right]}, "
                      f"area={current_area}, max_water={max_water}")
                
                if height[left] < height[right]:
                    left += 1
                else:
                    right -= 1
            
            return max_water
        
        # Example usage
        if __name__ == "__main__":
            # Example from problem description
            heights = [1,8,6,2,5,4,8,3,7]
            result = max_area(heights)
            print(f"\nMaximum water container area: {result}")
            
            # Show trace
            print("\nDetailed trace:")
            max_area_with_trace(heights)
        ```
        
        ## How it works with the example:
        
        For `height = [1,8,6,2,5,4,8,3,7]`:
        
        1. **Initial**: left=0 (height=1), right=8 (height=7) → area = min(1,7) × 8 = 8
        2. **Move left**: height[0]=1 < height[8]=7 → left=1
        3. **Step 2**: left=1 (height=8), right=8 (height=7) → area = min(8,7) × 7 = 49
        4. **Move right**: height[1]=8 > height[8]=7 → right=7
        5. **Continue** until pointers meet
        
        The algorithm efficiently finds the maximum area of 49 by systematically eliminating suboptimal possibilities.'''
        code_prompt2 = r'''I'll design an LRU Cache with O(1) time complexity for both get and put operations using a combination of a hash map and a doubly linked list.
        
        ```python
        class ListNode:
            """Doubly linked list node to store key-value pairs"""
            def __init__(self, key=0, value=0):
                self.key = key
                self.value = value
                self.prev = None
                self.next = None
        
        class LRUCache:
            """
            LRU Cache implementation with O(1) time complexity for get and put operations.
            
            Data Structures Used:
            1. Hash Map (Dictionary): For O(1) key lookup
            2. Doubly Linked List: For maintaining access order and O(1) insertion/deletion
            
            The cache maintains a doubly linked list where:
            - Most recently used items are near the head
            - Least recently used items are near the tail
            - Hash map maps keys to their corresponding list nodes
            """
            
            def __init__(self, capacity: int):
                """
                Initialize the LRU Cache with given capacity.
                
                Args:
                    capacity (int): Maximum number of items the cache can hold
                """
                self.capacity = capacity
                self.cache = {}  # Hash map: key -> ListNode
                
                # Create dummy head and tail nodes for easier list operations
                self.head = ListNode()
                self.tail = ListNode()
                self.head.next = self.tail
                self.tail.prev = self.head
            
            def _add_node(self, node: ListNode):
                """Add node right after head (most recently used position)"""
                node.prev = self.head
                node.next = self.head.next
                
                self.head.next.prev = node
                self.head.next = node
            
            def _remove_node(self, node: ListNode):
                """Remove an existing node from the linked list"""
                prev_node = node.prev
                next_node = node.next
                
                prev_node.next = next_node
                next_node.prev = prev_node
            
            def _move_to_head(self, node: ListNode):
                """Move existing node to head (mark as most recently used)"""
                self._remove_node(node)
                self._add_node(node)
            
            def _pop_tail(self) -> ListNode:
                """Remove and return the least recently used node (before tail)"""
                last_node = self.tail.prev
                self._remove_node(last_node)
                return last_node
            
            def get(self, key: int) -> int:
                """
                Get the value of the key if it exists in cache, otherwise return -1.
                
                Time Complexity: O(1)
                
                Args:
                    key (int): Key to look up
                    
                Returns:
                    int: Value if key exists, -1 otherwise
                """
                node = self.cache.get(key)
                if not node:
                    return -1
                
                # Move accessed node to head (most recently used)
                self._move_to_head(node)
                return node.value
            
            def put(self, key: int, value: int) -> None:
                """
                Insert or update the value of the key.
                
                Time Complexity: O(1)
                
                Args:
                    key (int): Key to insert/update
                    value (int): Value to associate with the key
                """
                node = self.cache.get(key)
                
                if not node:
                    # Key doesn't exist, create new node
                    new_node = ListNode(key, value)
                    
                    if len(self.cache) >= self.capacity:
                        # Cache is full, remove least recently used item
                        tail_node = self._pop_tail()
                        del self.cache[tail_node.key]
                    
                    # Add new node to cache and list
                    self.cache[key] = new_node
                    self._add_node(new_node)
                else:
                    # Key exists, update value and move to head
                    node.value = value
                    self._move_to_head(node)
        
        # Example usage and test cases
        if __name__ == "__main__":
            # Test case 1: Basic operations
            print("=== Test Case 1: Basic Operations ===")
            lru = LRUCache(2)
            
            lru.put(1, 1)
            lru.put(2, 2)
            print(f"get(1): {lru.get(1)}")  # returns 1
            
            lru.put(3, 3)  # evicts key 2
            print(f"get(2): {lru.get(2)}")  # returns -1 (not found)
            
            lru.put(4, 4)  # evicts key 1
            print(f"get(1): {lru.get(1)}")  # returns -1 (not found)
            print(f"get(3): {lru.get(3)}")  # returns 3
            print(f"get(4): {lru.get(4)}")  # returns 4
            
            # Test case 2: Update existing key
            print("\n=== Test Case 2: Update Existing Key ===")
            lru2 = LRUCache(2)
            lru2.put(1, 1)
            lru2.put(2, 2)
            print(f"get(1): {lru2.get(1)}")  # returns 1
            lru2.put(1, 10)  # update value
            print(f"get(1): {lru2.get(1)}")  # returns 10
            lru2.put(3, 3)  # evicts key 2
            print(f"get(2): {lru2.get(2)}")  # returns -1
            
            # Test case 3: Capacity boundary
            print("\n=== Test Case 3: Capacity Boundary ===")
            lru3 = LRUCache(3)
            lru3.put(1, 1)
            lru3.put(2, 2)
            lru3.put(3, 3)
            print(f"get(1): {lru3.get(1)}")  # returns 1
            lru3.put(4, 4)  # evicts key 2
            print(f"get(2): {lru3.get(2)}")  # returns -1
            print(f"get(1): {lru3.get(1)}")  # returns 1
            print(f"get(3): {lru3.get(3)}")  # returns 3
            print(f"get(4): {lru3.get(4)}")  # returns 4
        ```
        
        ## Design Explanation
        
        ### Data Structures Chosen:
        
        1. **Hash Map (Dictionary)**: 
           - **Why**: Provides O(1) average time complexity for key lookups
           - **Usage**: Maps keys to their corresponding list nodes for quick access
        
        2. **Doubly Linked List**:
           - **Why**: Allows O(1) insertion and deletion operations
           - **Usage**: Maintains the order of usage with most recently used at head and least recently used at tail
        
        ### Key Design Decisions:
        
        1. **Dummy Head and Tail Nodes**: 
           - Simplifies edge cases by avoiding null checks
           - Makes insertion/deletion operations uniform
        
        2. **Node Structure**:
           - Each node stores key, value, and pointers to prev/next nodes
           - Key is stored in the node to help with eviction
        
        3. **Operations**:
           - **get()**: O(1) - Look up in hash map, then move node to head
           - **put()**: O(1) - Either update existing node or create new, handle eviction if needed
        
        ### Time Complexity Analysis:
        - **get(key)**: O(1) - Hash map lookup + O(1) list operations
        - **put(key, value)**: O(1) - Hash map lookup + O(1) list operations + potential eviction
        
        ### Space Complexity:
        - O(capacity) for storing at most `capacity` items
        
        This implementation efficiently maintains the LRU property while ensuring both operations run in constant time.'''

        return reasoning_prompt1, reasoning_prompt2, logic_prompt1, logic_prompt2, code_prompt1, code_prompt2

    def count(self):

        gpt5_reasoning_prompt1, gpt5_reasoning_prompt2, gpt5_logic_prompt1, gpt5_logic_prompt2 = self.GPT5_response()
        
        gpt_reasoning_prompt1, gpt_reasoning_prompt2, gpt_logic_prompt1, gpt_logic_prompt2, gpt_code_prompt1, code_prompt2 = self.gpt_oss20b()
        qwen_reasoning_prompt1, qwen_reasoning_prompt2, qwen_logic_prompt1, qwen_logic_prompt2, qwen_code_prompt1, qwen_code_prompt2 = self.qwen330b()
        qwen_coder_reasoning_prompt1, qwen_coder_reasoning_prompt2, qwen_coder_logic_prompt1, qwen_coder_logic_prompt2, qwen_coder_code_prompt1, qwen_coder_code_prompt2 = self.Danielsheep_Qwen3_Coder()
        
        len_gpt5_reasoning_prompt1, len_gpt5_reasoning_prompt2, len_gpt5_logic_prompt1, len_gpt5_logic_prompt2 = len(gpt5_reasoning_prompt1), len(gpt5_reasoning_prompt2), len(gpt5_logic_prompt1), len(gpt5_logic_prompt2)
        len_gpt_reasoning_prompt1, len_gpt_reasoning_prompt2, len_gpt_logic_prompt1, len_gpt_logic_prompt2, len_gpt_code_prompt1, len_code_prompt2 = len(gpt_reasoning_prompt1), len(gpt_reasoning_prompt2), len(gpt_logic_prompt1), len(gpt_logic_prompt2), len(gpt_code_prompt1), len(code_prompt2)
        len_qwen_reasoning_prompt1, len_qwen_reasoning_prompt2, len_qwen_logic_prompt1, len_qwen_logic_prompt2, len_qwen_code_prompt1, len_qwen_code_prompt2 = len(qwen_reasoning_prompt1), len(qwen_reasoning_prompt2), len(qwen_logic_prompt1), len(qwen_logic_prompt2), len(qwen_code_prompt1), len(qwen_code_prompt2)
        len_qwen_coder_reasoning_prompt1, len_qwen_coder_reasoning_prompt2, len_qwen_coder_logic_prompt1, len_qwen_coder_logic_prompt2, len_qwen_coder_code_prompt1, len_qwen_coder_code_prompt2 = len(qwen_coder_reasoning_prompt1), len(qwen_coder_reasoning_prompt2), len(qwen_coder_logic_prompt1), len(qwen_coder_logic_prompt2), len(qwen_coder_code_prompt1), len(qwen_coder_code_prompt2)

        print("GPT5")
        print(len_gpt5_reasoning_prompt1, "\n", len_gpt5_reasoning_prompt2, "\n", len_gpt5_logic_prompt1, "\n", len_gpt5_logic_prompt2)
        print("\n Qwen3")
        print("GPT-OSS")
        print(len_gpt_reasoning_prompt1, "\n", len_gpt_reasoning_prompt2, "\n", len_gpt_logic_prompt1, "\n", len_gpt_logic_prompt2)
        print("\n Qwen3")
        print(len_qwen_reasoning_prompt1, "\n", len_qwen_reasoning_prompt2, "\n", len_qwen_logic_prompt1, "\n", len_qwen_logic_prompt2)
        print("\n Danielsheep_Qwen3_Coder")
        print(len_qwen_coder_reasoning_prompt1, "\n", len_qwen_coder_reasoning_prompt2, "\n", len_qwen_coder_logic_prompt1, "\n", len_qwen_coder_logic_prompt2)

    def extract_all_fenced_blocks(str, text: str) -> list[str]:
        pattern = r"```(?:python|py)\s*(.*?)```"
        return re.findall(pattern, text, flags=re.DOTALL)

    def normalize_code(self, code: str) -> str:
        code = code.expandtabs(4)
        lines = code.splitlines()
        normalized_lines = []

        indent_stack = []
        # Track if the previous non-empty line ended with a colon (block starter)
        last_line_opened_block = False

        for line in lines:
            if not line.strip():
                normalized_lines.append("")
                continue

            stripped = line.lstrip()
            raw_indent = len(line) - len(stripped)

            # 1. Initialize Stack
            if not indent_stack:
                indent_stack.append(raw_indent)

            # 2. Logic for Indentation Changes
            if raw_indent > indent_stack[-1]:
                # INDENT INCREASE
                # Filter: If this is a definition (def/class), only allow the indent to increase
                # if the previous line actually opened a block (ended in ':').
                # Otherwise, it's likely a copy-paste error (e.g., top-level def after an import).
                is_def_or_class = stripped.startswith(("def ", "class ", "async def ", "@"))

                if is_def_or_class and not last_line_opened_block: pass 
                else: indent_stack.append(raw_indent) # ALLOW the increase (nested block or multi-line statement)

            elif raw_indent < indent_stack[-1]:
                # INDENT DECREASE
                # Pop until we find the matching level
                while indent_stack and indent_stack[-1] > raw_indent:
                    indent_stack.pop()
                # If we went too far or landed on a mismatch, create a new anchor
                if not indent_stack or indent_stack[-1] != raw_indent:
                    indent_stack.append(raw_indent)

            # 3. Generate Output Line
            # Logical depth = stack size - 1 (because index 0 is the root anchor)
            logical_indent = max(0, len(indent_stack) - 1) * 4
            normalized_lines.append(" " * logical_indent + stripped)

            # 4. Update State for next iteration
            # Remove comments to check for trailing colon
            code_part = line.split("#")[0].strip()
            if code_part:
                last_line_opened_block = code_part.endswith(":")

        return "\n".join(normalized_lines).strip() + "\n"

    def get_similarity_score(self):
        import torch
        from sentence_transformers import SentenceTransformer, CrossEncoder, util
        
        # --- OPTIMIZATION FOR MAC M4 PRO ---
        device = "mps" if torch.backends.mps.is_available() else "cpu"
        print(f"Loading scoring models on: {device.upper()}...")

        # 1. Initialize Models
        embedder = SentenceTransformer('all-MiniLM-L6-v2', device=device)
        
        # NOTE: MS-MARCO is a Ranking model (1 output), not NLI (3 outputs)
        nli_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=device)
        
        # 2. Fetch Responses
        g5_r1, g5_r2, g5_l1, g5_l2 = self.GPT5_response()
        oss_r1, oss_r2, oss_l1, oss_l2, _, _ = self.gpt_oss20b()
        qwen_r1, qwen_r2, qwen_l1, qwen_l2, _, _ = self.qwen330b()
        coder_r1, coder_r2, coder_l1, coder_l2, _, _ = self.Danielsheep_Qwen3_Coder()

        # 3. Organize Data
        comparisons = [
            ("Reasoning 1", g5_r1, {"GPT-OSS": oss_r1, "Qwen3": qwen_r1, "Qwen-Coder": coder_r1}),
            ("Reasoning 2", g5_r2, {"GPT-OSS": oss_r2, "Qwen3": qwen_r2, "Qwen-Coder": coder_r2}),
            ("Logic 1",     g5_l1, {"GPT-OSS": oss_l1, "Qwen3": qwen_l1, "Qwen-Coder": coder_l1}),
            ("Logic 2",     g5_l2, {"GPT-OSS": oss_l2, "Qwen3": qwen_l2, "Qwen-Coder": coder_l2}),
        ]

        results = []
        
        print("\n" + "="*70)
        print(f"{'Task':<15} | {'Model':<15} | {'Cosine (Sim)':<12} | {'Relevance (0-1)':<15}")
        print("="*70)

        # 4. Calculate Scores
        for task, ref_text, candidates in comparisons:
            ref_emb = embedder.encode(ref_text, convert_to_tensor=True, device=device)

            for model_name, cand_text in candidates.items():
                
                # --- Metric A: Cosine Similarity ---
                cand_emb = embedder.encode(cand_text, convert_to_tensor=True, device=device)
                cosine_score = util.cos_sim(ref_emb, cand_emb).item()

                # --- Metric B: MS-MARCO Relevance ---
                # Predict returns a list of floats (raw logits) for MS-MARCO
                scores = nli_model.predict([(ref_text, cand_text)])
                
                # Take the single score (index 0)
                raw_score = scores[0]
                
                # Apply Sigmoid to convert raw score -> 0.0 to 1.0 probability
                relevance_score = torch.sigmoid(torch.tensor(raw_score)).item()

                print(f"{task:<15} | {model_name:<15} | {cosine_score:.4f}       | {relevance_score:.4f}")
                
                results.append({
                    "Task": task,
                    "Model": model_name,
                    "Cosine_Sim": cosine_score,
                    # Renamed to avoid confusion (this is now Relevance, not Entailment)
                    "Relevance_Score": relevance_score, 
                    "Generated_Text": cand_text[:50] + "..."
                })
        
        print("="*70)
        return results

    def get_code_benchmark(self):

        # --- OPTIMIZATION FOR MAC M4 PRO ---
        device = "mps" if torch.backends.mps.is_available() else "cpu"
        print(f"Loading Code Evaluation models on: {device.upper()}...")

        code_embedder = SentenceTransformer('./hf_models/models--nomic-ai--nomic-embed-code/snapshots/11114029805cee545ef111d5144b623787462a52', device=device)

        g5_c1, g5_c2 = self.Claude_code_logic_responses()

        # Get Candidates
        _, _, _, _, oss_c1, oss_c2 = self.gpt_oss20b()
        _, _, _, _, qwen_c1, qwen_c2 = self.qwen330b()
        _, _, _, _, coder_c1, coder_c2 = self.Danielsheep_Qwen3_Coder()

        comparisons = [
            ("Code 1", g5_c1, {"GPT-OSS": oss_c1, "Qwen3": qwen_c1, "Qwen-Coder": coder_c1}),
            ("Code 2", g5_c2, {"GPT-OSS": oss_c2, "Qwen3": qwen_c2, "Qwen-Coder": coder_c2}),
        ]

        results = []

        print("\n" + "="*80)
        print(f"{'Task':<10} | {'Model':<15} | {'CodeBERT (Sim)':<15} | {'Syntax (Valid)':<15}")
        print("="*80)

        for task, ref_code, candidates in comparisons:
            # Encode Reference
            ref_emb = code_embedder.encode(ref_code, convert_to_tensor=True, device=device)

            for model_name, cand_code in candidates.items():

                # # --- Metric A: CodeBERT Similarity ---
                cand_emb = code_embedder.encode(cand_code, convert_to_tensor=True, device=device)
                similarity = util.cos_sim(ref_emb, cand_emb).item()

                valid_python_blocks = []
                blocks = self.extract_all_fenced_blocks(cand_code)
                # --- Metric B: Syntax Validity (AST Check) ---
                for i, code in enumerate(blocks, 1):
                    try:
                        code = code.strip()
                        normalize_code = self.normalize_code(code)
                        ast.parse(normalize_code)
                        valid_python_blocks.append(0.9)
                    except SyntaxError: valid_python_blocks.append(0.1)
                    except Exception: valid_python_blocks.append(0)

                syntax_score = sum(valid_python_blocks) / len(valid_python_blocks) if valid_python_blocks else 0.0
                print(f"{task:<10} | {model_name:<15} | {similarity:.4f}          | {syntax_score:.4f}")

                results.append({
                    "Task": task,
                    "Model": model_name,
                    "Code_Similarity": similarity,
                    "Syntax_Pass": syntax_score,
                    "Code_Snippet": cand_code[:40].replace("\n", " ") + "..."
                })
        print("="*80)
        return results



In [61]:
ALLResponses().count()

GPT5
306 
 521 
 946 
 1070

 Qwen3
GPT-OSS
948 
 1620 
 2467 
 1991

 Qwen3
1400 
 2848 
 1473 
 2040

 Danielsheep_Qwen3_Coder
958 
 1278 
 1398 
 1712


In [26]:
similarity_scores = ALLResponses().get_similarity_score()
similarity_scores

Loading scoring models on: MPS...


Loading weights: 100%|██████████████████████████████████████| 103/103 [00:00<00:00, 1877.44it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|████████████████████████████████████████| 105/105 [00:00<00:00, 1824.54it/s, Materializing param=classifier.weight]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Task            | Model           | Cosine (Sim) | Relevance (0-1)
Reasoning 1     | GPT-OSS         | 0.7862       | 0.9579
Reasoning 1     | Qwen3           | 0.8373       | 0.9889
Reasoning 1     | Qwen-Coder      | 0.7827       | 0.9906
Reasoning 2     | GPT-OSS         | 0.8572       | 0.9955
Reasoning 2     | Qwen3           | 0.8329       | 0.9902
Reasoning 2     | Qwen-Coder      | 0.8821       | 0.9966
Logic 1         | GPT-OSS         | 0.7182       | 0.9939
Logic 1         | Qwen3           | 0.8371       | 0.9954
Logic 1         | Qwen-Coder      | 0.8484       | 0.9982
Logic 2         | GPT-OSS         | 0.8128       | 0.7967
Logic 2         | Qwen3           | 0.8398       | 0.9902
Logic 2         | Qwen-Coder      | 0.9140       | 0.9623


[{'Task': 'Reasoning 1',
  'Model': 'GPT-OSS',
  'Cosine_Sim': 0.7861843109130859,
  'Relevance_Score': 0.9579175114631653,
  'Generated_Text': '**Step‑by‑step solution**\n\n        1. **Define the...'},
 {'Task': 'Reasoning 1',
  'Model': 'Qwen3',
  'Cosine_Sim': 0.8372521996498108,
  'Relevance_Score': 0.9888889193534851,
  'Generated_Text': 'To solve the problem, we begin by defining the cos...'},
 {'Task': 'Reasoning 1',
  'Model': 'Qwen-Coder',
  'Cosine_Sim': 0.7827149629592896,
  'Relevance_Score': 0.9905773401260376,
  'Generated_Text': 'Let me solve this step-by-step.\n\n        Let me de...'},
 {'Task': 'Reasoning 2',
  'Model': 'GPT-OSS',
  'Cosine_Sim': 0.857249915599823,
  'Relevance_Score': 0.9955323934555054,
  'Generated_Text': '**Location:**  \n        The diamond is still in th...'},
 {'Task': 'Reasoning 2',
  'Model': 'Qwen3',
  'Cosine_Sim': 0.832929253578186,
  'Relevance_Score': 0.9902473092079163,
  'Generated_Text': 'Based on the properties of diamonds, glass, m

In [27]:
import pandas as pd

df = pd.DataFrame(similarity_scores)

In [29]:
import altair as alt

# Melt the data to make it "Tidy" (Long format) for Altair
df_long = df.melt(id_vars=['Task', 'Model'], 
                  value_vars=['Cosine_Sim', 'Relevance_Score'], 
                  var_name='Metric', value_name='Score')

# Create a Faceted Chart
chart = alt.Chart(df_long).mark_bar().encode(
    x=alt.X('Model:N', axis=None), # Hide X labels to avoid clutter
    y=alt.Y('Score:Q', scale=alt.Scale(domain=[0, 1])),
    color='Model:N',
    column='Task:N', # Split by Task horizontally
    row='Metric:N',  # Split by Metric vertically
    tooltip=['Task', 'Model', 'Metric', 'Score']
).properties(
    width=100,
    height=200,
    title="Benchmark Comparison"
).configure_view(
    stroke=None
)

chart.save('chart.html')

In [20]:
import pandas as pd
import plotly.express as px


# 2. Calculate the "Overall Score" (Mean of Entailment)
leaderboard = df.groupby('Model')['Entailment_Score'].mean().reset_index()
leaderboard['Overall_Score'] = (leaderboard['Entailment_Score'] * 100).round(2) # Convert to %
leaderboard = leaderboard.sort_values(by='Overall_Score', ascending=False)

# 3. Define the Winner
winner_name = leaderboard.iloc[0]['Model']
winner_score = leaderboard.iloc[0]['Overall_Score']

print("\n--- FINAL LEADERBOARD ---")
print(leaderboard[['Model', 'Overall_Score']].to_string(index=False))


--- FINAL LEADERBOARD ---
     Model  Overall_Score
Qwen-Coder          37.13
   GPT-OSS          25.13
     Qwen3          24.59


In [64]:
code_scores = ALLResponses().get_code_benchmark()
code_scores

Loading Code Evaluation models on: MPS...


Loading weights: 100%|███████████████████████████████████████████████████████████████████████| 338/338 [00:00<00:00, 2045.59it/s, Materializing param=norm.weight]



Task       | Model           | CodeBERT (Sim)  | Syntax (Valid) 
Code 1     | GPT-OSS         | 0.8584          | 0.9000
Code 1     | Qwen3           | 0.8942          | 0.9000
Code 1     | Qwen-Coder      | 0.9268          | 0.9000
Code 2     | GPT-OSS         | 0.8433          | 0.9000
Code 2     | Qwen3           | 0.8068          | 0.9000
Code 2     | Qwen-Coder      | 0.8399          | 0.9000


[{'Task': 'Code 1',
  'Model': 'GPT-OSS',
  'Code_Similarity': 0.8584023714065552,
  'Syntax_Pass': 0.9,
  'Code_Snippet': '**Explanation – Why a Two‑Pointer Scan W...'},
 {'Task': 'Code 1',
  'Model': 'Qwen3',
  'Code_Similarity': 0.8942462801933289,
  'Syntax_Pass': 0.9,
  'Code_Snippet': 'To solve the "Container With Most Water"...'},
 {'Task': 'Code 1',
  'Model': 'Qwen-Coder',
  'Code_Similarity': 0.9267607927322388,
  'Syntax_Pass': 0.9,
  'Code_Snippet': '## Logic Explanation          This is a ...'},
 {'Task': 'Code 2',
  'Model': 'GPT-OSS',
  'Code_Similarity': 0.8433372378349304,
  'Syntax_Pass': 0.9,
  'Code_Snippet': '**Solution Overview**          An LRU (L...'},
 {'Task': 'Code 2',
  'Model': 'Qwen3',
  'Code_Similarity': 0.806790828704834,
  'Syntax_Pass': 0.9,
  'Code_Snippet': 'To implement an LRU Cache with O(1) time...'},
 {'Task': 'Code 2',
  'Model': 'Qwen-Coder',
  'Code_Similarity': 0.8399232625961304,
  'Syntax_Pass': 0.9,
  'Code_Snippet': "I'll design an LRU C